# SafeHer AI — Model Training Notebook
> End-to-end pipeline: dataset generation → preprocessing → model training → evaluation

**Author:** Prem | **Project:** SafeHer AI | **Stack:** Python, Scikit-learn, XGBoost, Pandas

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports successful')

## 1. Generate Synthetic Dataset

In [ ]:
from src.data_generator import generate_dataset
df = generate_dataset(n_rows=10_000, save_path='../data/safety_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Risk Level Distribution:')
print(df['risk_level'].value_counts())

fig = px.pie(df, names='risk_level', title='Risk Level Distribution',
             color='risk_level',
             color_discrete_map={'Safe':'#22c55e','Medium Risk':'#f59e0b',
                                  'High Risk':'#ef4444','Emergency':'#a855f7'})
fig.show()

In [ ]:
fig2 = px.histogram(df, x='total_risk_score', color='risk_level',
                    nbins=50, title='Risk Score Distribution by Level',
                    color_discrete_map={'Safe':'#22c55e','Medium Risk':'#f59e0b',
                                        'High Risk':'#ef4444','Emergency':'#a855f7'})
fig2.show()

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()
fig3, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, cmap='RdPu', center=0, ax=ax, fmt='.2f', annot=False, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Preprocess Data

In [ ]:
from src.preprocessing import preprocess
X_train, X_test, y_train, y_test, features, encoders, scaler = preprocess(df, fit=True, models_dir='../models')
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features ({len(features)}):', features[:8], '...')

## 4. Train All Models

In [ ]:
from src.train_model import train_all_models
results, best_name, best_model = train_all_models(
    X_train, X_test, y_train, y_test, features, models_dir='../models'
)
print(f'Best model: {best_name}')

## 5. Visualise Model Comparison

In [ ]:
model_names = list(results.keys())
accs = [results[m]['accuracy']    for m in model_names]
f1s  = [results[m]['f1_weighted'] for m in model_names]

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(name='Accuracy',     x=model_names, y=accs, marker_color='#6366f1'))
fig_bar.add_trace(go.Bar(name='F1 (Weighted)',x=model_names, y=f1s,  marker_color='#ec4899'))
fig_bar.update_layout(barmode='group', title='Model Comparison', yaxis_range=[0, 1.1])
fig_bar.show()

## 6. Confusion Matrix — Best Model

In [ ]:
import seaborn as sns
cm = results[best_name]['confusion_matrix']
labels = ['Safe', 'Medium Risk', 'High Risk', 'Emergency']
fig_cm, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13)
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
fi = results[best_name].get('feature_importances', {})
if fi:
    fi_sorted = sorted(fi.items(), key=lambda x: x[1], reverse=True)[:15]
    fig_fi = px.bar(x=[v for _, v in fi_sorted], y=[k for k, _ in fi_sorted],
                    orientation='h', title=f'Feature Importance — {best_name}',
                    color=[v for _, v in fi_sorted], color_continuous_scale='RdPu')
    fig_fi.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig_fi.show()
else:
    print('Feature importances not available for this model type.')

## 8. Test a Sample Prediction

In [ ]:
from src.predict import SafetyPredictor
predictor = SafetyPredictor('../models')

sample = {
    'latitude': 28.6139, 'longitude': 77.2090,
    'hour': 23, 'day_type': 1, 'is_night': 1,
    'is_alone': 1, 'crowd_density': 'low',
    'lighting_condition': 'dark', 'crime_area_score': 0.75,
    'phone_motion_status': 'erratic', 'walking_speed': 5.5,
    'distance_from_home': 12.0, 'distance_from_safe_zone': 8.0,
    'battery_level': 18, 'network_status': '2G',
    'weather_condition': 'foggy', 'panic_button': 0,
    'unusual_movement': 1, 'time_spent_at_location': 45,
    'nearest_trusted_contact_distance': 20.0,
}
result = predictor.predict(sample)
print(f"Risk Level : {result['icon']} {result['risk_level']}")
print(f"Risk Score : {result['risk_score']} / 100")
print(f"Probabilities: {result['probabilities']}")
print('Suggestions:')
for s in result['suggestions']:
    print(f'  {s}')

## ✅ Notebook Complete
Model trained and saved to `../models/safety_model.pkl`. Run `streamlit run app.py` to launch the dashboard.